In [0]:
configs = {"fs.azure.account.auth.type": "OAuth",
"fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
"fs.azure.account.oauth2.client.id": "2fc62fb4-f603-4856-8af3-dfb6b7ab0dd6",
"fs.azure.account.oauth2.client.secret": "YOUR_CLIENT_SECRET",
"fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/dde4ab60-a7f5-4520-9560-518d4b424ed4/oauth2/token"}

source = "abfss://retail@deretailproject.dfs.core.windows.net"  # contrainer@storageacc

for key, value in configs.items():
    spark.conf.set(key, value)

source

'abfss://retail@deretailproject.dfs.core.windows.net'

In [0]:
display(dbutils.fs.ls(source))

path,name,size,modificationTime
abfss://retail@deretailproject.dfs.core.windows.net/bronze/,bronze/,0,1788110478000
abfss://retail@deretailproject.dfs.core.windows.net/gold/,gold/,0,1788110490000
abfss://retail@deretailproject.dfs.core.windows.net/silver/,silver/,0,1788110484000


In [0]:
products = (spark.read.format("parquet").option("header", "true").option("inferSchema", "true").load("abfss://retail@deretailproject.dfs.core.windows.net/bronze/product/dbo.products.parquet"))
stores = (spark.read.format("parquet").option("header", "true").option("inferSchema", "true").load("abfss://retail@deretailproject.dfs.core.windows.net/bronze/store/dbo.stores.parquet"))
transactions = (spark.read.format("parquet").option("header", "true").option("inferSchema", "true").load("abfss://retail@deretailproject.dfs.core.windows.net/bronze/transaction/dbo.transactions.parquet"))
customers = (spark.read.format("parquet").option("header", "true").option("inferSchema", "true").load("abfss://retail@deretailproject.dfs.core.windows.net/bronze/customer/manish040596/azure-data-engineer---multi-source/refs/heads/main/customers.parquet"))

In [0]:
display(products)

In [0]:
display(stores)

In [0]:
display(transactions)

In [0]:
display(customers)

In [0]:
from pyspark.sql.functions import col

# Convert types and clean data
transactions = transactions.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date")
)


In [0]:
transactions.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)



In [0]:
products = products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

In [0]:
products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [0]:
stores = stores.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)

In [0]:
stores.printSchema()

root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- location: string (nullable = true)



In [0]:
customers = customers.select(
    "customer_id", "first_name", "last_name", "email", "city", "registration_date"
).dropDuplicates(["customer_id"])

In [0]:
silver = transactions \
    .join(customers, "customer_id") \
    .join(products, "product_id") \
    .join(stores, "store_id") \
    .withColumn("total_amount", col("quantity") * col("price"))

In [0]:
display(silver)

In [0]:
silver.write.mode("overwrite").format("delta").save(source + "/silver/")

In [0]:
silver = spark.read.format("delta").load(
    "abfss://retail@deretailproject.dfs.core.windows.net/silver/"
)

display(silver)

In [0]:
from pyspark.sql.functions import sum, countDistinct, avg

gold = silver.groupBy(
    "transaction_date",
    "product_id", "product_name", "category",
    "store_id", "store_name", "location"
).agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("total_amount").alias("total_sales_amount"),
    countDistinct("transaction_id").alias("number_of_transactions"),
    avg("total_amount").alias("average_transaction_value")
)


In [0]:
display(gold)

In [0]:
gold.write.mode("overwrite").format("delta").save(source + "/gold/")

In [0]:
gold = spark.read.format("delta").load(
    "abfss://retail@deretailproject.dfs.core.windows.net/gold/"
)

display(gold)